Optimization & Recommendations

In [18]:
# setup: paths, source validation, mapping discovery, and reusable helpers

from pathlib import Path
import hashlib
import warnings
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\hp\Nassau_Candy_Final_Submission")
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "Nassau Candy Distributor.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "optimization_recommendations"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

for folder in [PROCESSED_DIR, REPORT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Original CSV not found: {RAW_FILE}")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ORIGINAL_HASH_BEFORE = sha256_file(RAW_FILE)
df_phase10 = pd.read_csv(RAW_FILE)
original_columns = df_phase10.columns.tolist()
original_row_count = len(df_phase10)

required_columns = [
    "Product Name", "Order Date", "Ship Date", "Sales",
    "Units", "Gross Profit", "Cost"
]
missing_columns = [c for c in required_columns if c not in df_phase10.columns]
if missing_columns:
    raise ValueError(f"Required source columns missing: {missing_columns}")

# Find the Phase 07 product-factory mapping without assuming a specific filename.
mapping_candidates = [
    PROCESSED_DIR / "product_factory_mapping.csv",
    PROCESSED_DIR / "Product_Factory_Mapping.csv",
    PROCESSED_DIR / "factory_product_mapping.csv"
]
MAPPING_FILE = next((p for p in mapping_candidates if p.exists()), None)

if MAPPING_FILE is None:
    candidates = list(PROCESSED_DIR.glob("*mapping*.csv")) + list(PROCESSED_DIR.glob("*Mapping*.csv"))
    print("Possible mapping files found:")
    for candidate in candidates:
        print(" -", candidate.name)
    raise FileNotFoundError(
        "Could not identify the Phase 07 product-factory mapping. "
        "Check the filenames printed above and update MAPPING_FILE."
    )

mapping_raw = pd.read_csv(MAPPING_FILE)
print("Mapping file:", MAPPING_FILE.name)
print("Mapping columns:", mapping_raw.columns.tolist())

def find_column(columns, candidates, description):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise ValueError(f"Could not identify {description}. Available columns: {list(columns)}")

PRODUCT_MAP_COL = find_column(
    mapping_raw.columns,
    ["Product Name", "Product", "product_name"],
    "product-name column"
)
FACTORY_MAP_COL = find_column(
    mapping_raw.columns,
    ["Factory Name", "Mapped Factory", "Factory", "factory_name", "Mapped Factory Name"],
    "factory-name column"
)

mapping = mapping_raw[[PRODUCT_MAP_COL, FACTORY_MAP_COL]].copy()
mapping.columns = ["Product Name", "Reference Factory"]
mapping = mapping.dropna().drop_duplicates()

if mapping["Product Name"].duplicated().any():
    raise ValueError("The mapping contains multiple reference factories for at least one product.")

source_products = set(df_phase10["Product Name"].dropna().unique())
mapped_products = set(mapping["Product Name"].dropna().unique())
unmapped_products = sorted(source_products - mapped_products)

# Derive the historical date difference without changing source date strings.
order_dates = pd.to_datetime(df_phase10["Order Date"], dayfirst=True, errors="coerce")
ship_dates = pd.to_datetime(df_phase10["Ship Date"], dayfirst=True, errors="coerce")
df_phase10["_Derived_Lead_Time_Days"] = (ship_dates - order_dates).dt.days

# Numeric analysis copies; original columns remain untouched.
for col in ["Sales", "Units", "Gross Profit", "Cost"]:
    df_phase10[f"_{col}_Numeric"] = pd.to_numeric(df_phase10[col], errors="coerce")

# Create a blank hypothetical incremental-cost input template.
HYPOTHETICAL_COST_FILE = PROCESSED_DIR / "phase10_hypothetical_cost_inputs_template.csv"
if not HYPOTHETICAL_COST_FILE.exists():
    cost_template = mapping.rename(columns={"Reference Factory": "Factory Option"}).copy()
    cost_template["Hypothetical_Additional_Cost_Per_Unit"] = np.nan
    cost_template["Assumption_Notes"] = ""
    cost_template["Cost_Type"] = "Hypothetical incremental cost per unit only"
    cost_template.to_csv(HYPOTHETICAL_COST_FILE, index=False)

print("Original rows:", original_row_count)
print("Original columns:", len(original_columns))
print("Products in source:", len(source_products))
print("Products in mapping:", len(mapped_products))
print("Unmapped products:", unmapped_products)
print("Reference factories:", sorted(mapping["Reference Factory"].unique()))
print("Original SHA-256:", ORIGINAL_HASH_BEFORE)
print("Hypothetical cost template:", HYPOTHETICAL_COST_FILE)

Mapping file: product_factory_mapping.csv
Mapping columns: ['Product Name', 'Mapped Factory', 'Reference Division']
Original rows: 10194
Original columns: 18
Products in source: 15
Products in mapping: 15
Unmapped products: []
Reference factories: ["Lot's O' Nuts", 'Secret Factory', 'Sugar Shack', 'The Other Factory', "Wicked Choccy's"]
Original SHA-256: 6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f
Hypothetical cost template: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\phase10_hypothetical_cost_inputs_template.csv


Eligibility, options, baseline, time, variability, and profitability

In [21]:
# Define eligible products

# Define product eligibility using source coverage and reference-mapping availability

product_eligibility = pd.DataFrame({"Product Name": sorted(source_products)})
product_eligibility = product_eligibility.merge(mapping, on="Product Name", how="left")

product_eligibility["Source_Data_Available"] = product_eligibility["Product Name"].isin(source_products)
product_eligibility["Reference_Mapping_Available"] = product_eligibility["Reference Factory"].notna()
product_eligibility["Eligibility_Status"] = np.where(
    product_eligibility["Source_Data_Available"] & product_eligibility["Reference_Mapping_Available"],
    "Eligible for product-level screening",
    "Flagged - incomplete mapping or source support"
)
product_eligibility["Factory_Recommendation_Supported"] = False
product_eligibility["Eligibility_Limitation"] = (
    "Eligibility supports product-level review only; it does not establish alternative-factory compatibility."
)

product_eligibility.to_csv(REPORT_DIR / "task_10_01_product_eligibility.csv", index=False)
display(product_eligibility)

,Product Name,Reference Factory,Source_Data_Available,Reference_Mapping_Available,Eligibility_Status,Factory_Recommendation_Supported,Eligibility_Limitation
0,Everlasting Gobstopper,Secret Factory,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
1,Fizzy Lifting Drinks,Sugar Shack,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
2,Fun Dip,Sugar Shack,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
3,Hair Toffee,The Other Factory,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
4,Kazookles,The Other Factory,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
5,Laffy Taffy,Sugar Shack,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
6,Lickable Wallpaper,Secret Factory,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
7,Nerds,Sugar Shack,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
8,SweeTARTS,Sugar Shack,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...
9,Wonka Bar - Fudge Mallows,Lot's O' Nuts,True,True,Eligible for product-level screening,False,Eligibility supports product-level review only...


In [23]:
# Generate alternative factory options

# Generate alternative factory screening options without asserting compatibility

reference_factories = sorted(mapping["Reference Factory"].dropna().unique())
eligible_products = product_eligibility.loc[
    product_eligibility["Eligibility_Status"] == "Eligible for product-level screening",
    ["Product Name", "Reference Factory"]
].copy()

factory_options = eligible_products.assign(_key=1).merge(
    pd.DataFrame({"Factory Option": reference_factories, "_key": 1}),
    on="_key"
).drop(columns="_key")

factory_options["Is_Reference_Mapping"] = (
    factory_options["Factory Option"] == factory_options["Reference Factory"]
)
factory_options["Option_Type"] = np.where(
    factory_options["Is_Reference_Mapping"],
    "Reference mapping only - not verified historical dispatch",
    "Alternative factory - unverified"
)
factory_options["Compatibility_Status"] = np.where(
    factory_options["Is_Reference_Mapping"],
    "Reference mapping; actual dispatch not verified",
    "UNVERIFIED"
)
factory_options["Factory_Specific_Lead_Time_Estimable"] = False
factory_options["Factory_Specific_Cost_Available"] = False
factory_options["Capacity_Data_Available"] = False

factory_options.to_csv(REPORT_DIR / "task_10_02_factory_options.csv", index=False)
display(factory_options.head(20))
print("Total product-factory combinations:", len(factory_options))

,Product Name,Reference Factory,Factory Option,Is_Reference_Mapping,Option_Type,Compatibility_Status,Factory_Specific_Lead_Time_Estimable,Factory_Specific_Cost_Available,Capacity_Data_Available
0,Everlasting Gobstopper,Secret Factory,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False
1,Everlasting Gobstopper,Secret Factory,Secret Factory,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False
2,Everlasting Gobstopper,Secret Factory,Sugar Shack,False,Alternative factory - unverified,UNVERIFIED,False,False,False
3,Everlasting Gobstopper,Secret Factory,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False
4,Everlasting Gobstopper,Secret Factory,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False
5,Fizzy Lifting Drinks,Sugar Shack,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False
6,Fizzy Lifting Drinks,Sugar Shack,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False
7,Fizzy Lifting Drinks,Sugar Shack,Sugar Shack,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False
8,Fizzy Lifting Drinks,Sugar Shack,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False
9,Fizzy Lifting Drinks,Sugar Shack,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False


Total product-factory combinations: 75


In [25]:
# Create the historical baseline comparison

# Build product-level historical baseline measures from the unchanged source rows

baseline = (
    df_phase10.groupby("Product Name", dropna=False)
    .agg(
        Order_Line_Count=("Product Name", "size"),
        Historical_Sales=("_Sales_Numeric", "sum"),
        Historical_Units=("_Units_Numeric", "sum"),
        Historical_Gross_Profit=("_Gross Profit_Numeric", "sum"),
        Historical_Cost=("_Cost_Numeric", "sum"),
        Derived_Lead_Time_Mean_Days=("_Derived_Lead_Time_Days", "mean"),
        Derived_Lead_Time_Median_Days=("_Derived_Lead_Time_Days", "median"),
        Derived_Lead_Time_Min_Days=("_Derived_Lead_Time_Days", "min"),
        Derived_Lead_Time_Max_Days=("_Derived_Lead_Time_Days", "max")
    )
    .reset_index()
)

baseline["Historical_Gross_Margin"] = np.where(
    baseline["Historical_Sales"] != 0,
    baseline["Historical_Gross_Profit"] / baseline["Historical_Sales"],
    np.nan
)
baseline["Lead_Time_Measure_Label"] = "Derived Order Date to Ship Date difference; not verified transit duration"
baseline = baseline.merge(mapping, on="Product Name", how="left")

baseline.to_csv(REPORT_DIR / "task_10_03_historical_baseline.csv", index=False)
display(baseline)

,Product Name,Order_Line_Count,Historical_Sales,Historical_Units,Historical_Gross_Profit,Historical_Cost,Derived_Lead_Time_Mean_Days,Derived_Lead_Time_Median_Days,Derived_Lead_Time_Min_Days,Derived_Lead_Time_Max_Days,Historical_Gross_Margin,Lead_Time_Measure_Label,Reference Factory
0,Everlasting Gobstopper,3,130.00,13,104.00,26.00,1394.666667,1273.0,1270,1641,0.800000,Derived Order Date to Ship Date difference; no...,Secret Factory
1,Fizzy Lifting Drinks,6,78.75,21,47.25,31.50,1273.500000,1274.0,907,1640,0.600000,Derived Order Date to Ship Date difference; no...,Sugar Shack
2,Fun Dip,3,12.00,8,4.80,7.20,1272.666667,1272.0,1271,1275,0.400000,Derived Order Date to Ship Date difference; no...,Sugar Shack
3,Hair Toffee,4,76.50,17,59.50,17.00,1455.250000,1455.0,1272,1639,0.777778,Derived Order Date to Ship Date difference; no...,The Other Factory
4,Kazookles,96,1205.75,371,92.75,1113.00,1272.989583,1273.0,904,1641,0.076923,Derived Order Date to Ship Date difference; no...,The Other Factory
5,Laffy Taffy,10,53.73,27,33.48,20.25,1383.100000,1275.0,909,1642,0.623116,Derived Order Date to Ship Date difference; no...,Sugar Shack
6,Lickable Wallpaper,94,7860.00,393,3930.00,3930.00,1339.138298,1274.0,904,1641,0.500000,Derived Order Date to Ship Date difference; no...,Secret Factory
7,Nerds,4,15.00,10,7.00,8.00,1274.000000,1274.0,908,1640,0.466667,Derived Order Date to Ship Date difference; no...,Sugar Shack
8,SweeTARTS,10,61.50,41,28.70,32.80,1383.500000,1274.0,910,1641,0.466667,Derived Order Date to Ship Date difference; no...,Sugar Shack
9,Wonka Bar - Fudge Mallows,1818,24890.40,6914,16593.60,8296.80,1314.242024,1274.0,904,1642,0.666667,Derived Order Date to Ship Date difference; no...,Lot's O' Nuts


In [27]:
# Compare available lead-time estimates

# Compare historical derived lead-time summaries; alternative-factory estimates remain unavailable

time_comparison = factory_options.merge(
    baseline[[
        "Product Name", "Derived_Lead_Time_Mean_Days",
        "Derived_Lead_Time_Median_Days", "Derived_Lead_Time_Min_Days",
        "Derived_Lead_Time_Max_Days"
    ]],
    on="Product Name",
    how="left"
)

time_comparison["Factory_Option_Lead_Time_Estimate_Days"] = np.nan
time_comparison["Estimated_Lead_Time_Change_Days"] = np.nan
time_comparison["Time_Comparison_Status"] = np.where(
    time_comparison["Is_Reference_Mapping"],
    "Historical product baseline only; not a factory-specific estimate",
    "NOT ESTIMABLE - no validated factory effect or actual factory assignment"
)
time_comparison["Time_Ranking"] = "Not ranked by factory-specific lead time"

time_comparison.to_csv(REPORT_DIR / "task_10_04_lead_time_comparison.csv", index=False)
display(time_comparison.head(20))

,Product Name,Reference Factory,Factory Option,Is_Reference_Mapping,Option_Type,Compatibility_Status,Factory_Specific_Lead_Time_Estimable,Factory_Specific_Cost_Available,Capacity_Data_Available,Derived_Lead_Time_Mean_Days,Derived_Lead_Time_Median_Days,Derived_Lead_Time_Min_Days,Derived_Lead_Time_Max_Days,Factory_Option_Lead_Time_Estimate_Days,Estimated_Lead_Time_Change_Days,Time_Comparison_Status,Time_Ranking
0,Everlasting Gobstopper,Secret Factory,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1394.666667,1273.0,1270,1641,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
1,Everlasting Gobstopper,Secret Factory,Secret Factory,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False,1394.666667,1273.0,1270,1641,NaN,NaN,Historical product baseline only; not a factor...,Not ranked by factory-specific lead time
2,Everlasting Gobstopper,Secret Factory,Sugar Shack,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1394.666667,1273.0,1270,1641,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
3,Everlasting Gobstopper,Secret Factory,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1394.666667,1273.0,1270,1641,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
4,Everlasting Gobstopper,Secret Factory,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1394.666667,1273.0,1270,1641,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
5,Fizzy Lifting Drinks,Sugar Shack,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1273.500000,1274.0,907,1640,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
6,Fizzy Lifting Drinks,Sugar Shack,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1273.500000,1274.0,907,1640,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
7,Fizzy Lifting Drinks,Sugar Shack,Sugar Shack,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False,1273.500000,1274.0,907,1640,NaN,NaN,Historical product baseline only; not a factor...,Not ranked by factory-specific lead time
8,Fizzy Lifting Drinks,Sugar Shack,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1273.500000,1274.0,907,1640,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time
9,Fizzy Lifting Drinks,Sugar Shack,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1273.500000,1274.0,907,1640,NaN,NaN,NOT ESTIMABLE - no validated factory effect or...,Not ranked by factory-specific lead time


In [29]:
# Compare historical variability

# Calculate descriptive historical lead-time variability by product

variability = (
    df_phase10.groupby("Product Name")["_Derived_Lead_Time_Days"]
    .agg(
        Lead_Time_Std_Days="std",
        Lead_Time_IQR_Days=lambda s: s.quantile(0.75) - s.quantile(0.25),
        Valid_Lead_Time_Records="count"
    )
    .reset_index()
)

variability["Variability_Interpretation"] = (
    "Descriptive historical variation only; not a measure of factory risk or causal factory performance"
)
variability["Variability_Risk_Proxy"] = variability["Lead_Time_Std_Days"]

variability.to_csv(REPORT_DIR / "task_10_05_historical_variability.csv", index=False)
display(variability)

,Product Name,Lead_Time_Std_Days,Lead_Time_IQR_Days,Valid_Lead_Time_Records,Variability_Interpretation,Variability_Risk_Proxy
0,Everlasting Gobstopper,213.336198,185.50,3,Descriptive historical variation only; not a m...,213.336198
1,Fizzy Lifting Drinks,326.691751,546.00,6,Descriptive historical variation only; not a m...,326.691751
2,Fun Dip,2.081666,2.00,3,Descriptive historical variation only; not a m...,2.081666
3,Hair Toffee,209.876750,361.75,4,Descriptive historical variation only; not a m...,209.876750
4,Kazookles,259.943212,186.50,96,Descriptive historical variation only; not a m...,259.943212
5,Laffy Taffy,246.563695,365.00,10,Descriptive historical variation only; not a m...,246.563695
6,Lickable Wallpaper,288.459855,368.75,94,Descriptive historical variation only; not a m...,288.459855
7,Nerds,422.043442,730.50,4,Descriptive historical variation only; not a m...,422.043442
8,SweeTARTS,246.688850,366.50,10,Descriptive historical variation only; not a m...,246.688850
9,Wonka Bar - Fudge Mallows,264.102825,367.00,1818,Descriptive historical variation only; not a m...,264.102825


In [33]:
# Attach profitability indicators

# Attach historical profitability indicators and optional hypothetical cost inputs

profitability = baseline[[
    "Product Name", "Historical_Sales", "Historical_Units",
    "Historical_Gross_Profit", "Historical_Cost", "Historical_Gross_Margin"
]].copy()

profitability["Gross_Profit_Per_Unit"] = np.where(
    profitability["Historical_Units"] != 0,
    profitability["Historical_Gross_Profit"] / profitability["Historical_Units"],
    np.nan
)

hypothetical_cost_inputs = pd.read_csv(HYPOTHETICAL_COST_FILE)
hypothetical_cost_inputs["Hypothetical_Additional_Cost_Per_Unit"] = pd.to_numeric(
    hypothetical_cost_inputs["Hypothetical_Additional_Cost_Per_Unit"],
    errors="coerce"
)

profitability.to_csv(REPORT_DIR / "task_10_06_profitability_indicators.csv", index=False)
print("Hypothetical cost inputs are stored separately and blank by default.")
display(profitability)

Hypothetical cost inputs are stored separately and blank by default.


,Product Name,Historical_Sales,Historical_Units,Historical_Gross_Profit,Historical_Cost,Historical_Gross_Margin,Gross_Profit_Per_Unit
0,Everlasting Gobstopper,130.00,13,104.00,26.00,0.800000,8.00
1,Fizzy Lifting Drinks,78.75,21,47.25,31.50,0.600000,2.25
2,Fun Dip,12.00,8,4.80,7.20,0.400000,0.60
3,Hair Toffee,76.50,17,59.50,17.00,0.777778,3.50
4,Kazookles,1205.75,371,92.75,1113.00,0.076923,0.25
5,Laffy Taffy,53.73,27,33.48,20.25,0.623116,1.24
6,Lickable Wallpaper,7860.00,393,3930.00,3930.00,0.500000,10.00
7,Nerds,15.00,10,7.00,8.00,0.466667,0.70
8,SweeTARTS,61.50,41,28.70,32.80,0.466667,0.70
9,Wonka Bar - Fudge Mallows,24890.40,6914,16593.60,8296.80,0.666667,2.40


# Scoring design and ranking

In [36]:
# Define the recommendation scoring formula

In [38]:

# Define transparent product-level screening scores and document their meaning

scoring_formula = pd.DataFrame([
    ["Speed priority", "Higher historical mean derived lead-time receives higher review priority.", "Product-level screening only; not a predicted factory improvement."],
    ["Profit priority", "Higher historical gross margin receives higher profit-protection priority.", "Historical margin is not alternative-factory profitability."],
    ["Balanced score", "Weighted average of speed priority and profit priority.", "Weights are explicit and adjustable; score is not an optimization proof."],
    ["Factory option treatment", "The product-level score is copied across its factory options.", "No factory-specific score differences are inferred."],
    ["Hypothetical cost treatment", "Optional user-entered incremental cost per unit can support arithmetic sensitivity.", "Blank values mean no cost assumption; not actual factory cost."]
], columns=["Component", "Formula_or_Rule", "Interpretation_and_Limitation"])

scoring_formula.to_csv(REPORT_DIR / "task_10_07_scoring_formula.csv", index=False)
display(scoring_formula)

,Component,Formula_or_Rule,Interpretation_and_Limitation
0,Speed priority,Higher historical mean derived lead-time recei...,Product-level screening only; not a predicted ...
1,Profit priority,Higher historical gross margin receives higher...,Historical margin is not alternative-factory p...
2,Balanced score,Weighted average of speed priority and profit ...,Weights are explicit and adjustable; score is ...
3,Factory option treatment,The product-level score is copied across its f...,No factory-specific score differences are infe...
4,Hypothetical cost treatment,Optional user-entered incremental cost per uni...,Blank values mean no cost assumption; not actu...


In [40]:
# Calculate speed priority score

# Calculate a normalized product-level speed review priority score

speed_scores = baseline[["Product Name", "Derived_Lead_Time_Mean_Days"]].copy()

def minmax_score(series):
    series = pd.to_numeric(series, errors="coerce")
    minimum, maximum = series.min(), series.max()
    if pd.isna(minimum) or pd.isna(maximum):
        return pd.Series(np.nan, index=series.index)
    if maximum == minimum:
        return pd.Series(0.5, index=series.index)
    return (series - minimum) / (maximum - minimum)

speed_scores["Speed_Priority_Score"] = minmax_score(
    speed_scores["Derived_Lead_Time_Mean_Days"]
)
speed_scores["Score_Direction"] = "Higher means greater review priority due to longer historical derived lead time"
speed_scores.to_csv(REPORT_DIR / "task_10_08_speed_priority_scores.csv", index=False)
display(speed_scores)

,Product Name,Derived_Lead_Time_Mean_Days,Speed_Priority_Score,Score_Direction
0,Everlasting Gobstopper,1394.666667,0.668188,Higher means greater review priority due to lo...
1,Fizzy Lifting Drinks,1273.500000,0.004564,Higher means greater review priority due to lo...
2,Fun Dip,1272.666667,0.000000,Higher means greater review priority due to lo...
3,Hair Toffee,1455.250000,1.000000,Higher means greater review priority due to lo...
4,Kazookles,1272.989583,0.001769,Higher means greater review priority due to lo...
5,Laffy Taffy,1383.100000,0.604838,Higher means greater review priority due to lo...
6,Lickable Wallpaper,1339.138298,0.364062,Higher means greater review priority due to lo...
7,Nerds,1274.000000,0.007303,Higher means greater review priority due to lo...
8,SweeTARTS,1383.500000,0.607029,Higher means greater review priority due to lo...
9,Wonka Bar - Fudge Mallows,1314.242024,0.227706,Higher means greater review priority due to lo...


In [42]:
# Calculate profit priority score

# Calculate a normalized historical profit-protection priority score

profit_scores = profitability[["Product Name", "Historical_Gross_Margin"]].copy()

# Lower-margin products receive higher review priority.
profit_scores["Profit_Priority_Score"] = 1 - minmax_score(
    profit_scores["Historical_Gross_Margin"]
)
profit_scores["Score_Direction"] = "Higher means greater review priority due to lower historical gross margin"
profit_scores.to_csv(REPORT_DIR / "task_10_09_profit_priority_scores.csv", index=False)
display(profit_scores)

,Product Name,Historical_Gross_Margin,Profit_Priority_Score,Score_Direction
0,Everlasting Gobstopper,0.800000,0.000000,Higher means greater review priority due to lo...
1,Fizzy Lifting Drinks,0.600000,0.276596,Higher means greater review priority due to lo...
2,Fun Dip,0.400000,0.553191,Higher means greater review priority due to lo...
3,Hair Toffee,0.777778,0.030733,Higher means greater review priority due to lo...
4,Kazookles,0.076923,1.000000,Higher means greater review priority due to lo...
5,Laffy Taffy,0.623116,0.244627,Higher means greater review priority due to lo...
6,Lickable Wallpaper,0.500000,0.414894,Higher means greater review priority due to lo...
7,Nerds,0.466667,0.460993,Higher means greater review priority due to lo...
8,SweeTARTS,0.466667,0.460993,Higher means greater review priority due to lo...
9,Wonka Bar - Fudge Mallows,0.666667,0.184397,Higher means greater review priority due to lo...


In [44]:
# Calculate balanced product-level score

# Combine speed and profit priorities using explicit default weights

DEFAULT_SPEED_WEIGHT = 0.50
DEFAULT_PROFIT_WEIGHT = 0.50

if not np.isclose(DEFAULT_SPEED_WEIGHT + DEFAULT_PROFIT_WEIGHT, 1.0):
    raise ValueError("Default weights must sum to 1.")

product_scores = speed_scores.merge(profit_scores, on="Product Name", how="outer")
product_scores["Balanced_Score"] = (
    DEFAULT_SPEED_WEIGHT * product_scores["Speed_Priority_Score"]
    + DEFAULT_PROFIT_WEIGHT * product_scores["Profit_Priority_Score"]
)
product_scores["Speed_Weight"] = DEFAULT_SPEED_WEIGHT
product_scores["Profit_Weight"] = DEFAULT_PROFIT_WEIGHT
product_scores["Score_Interpretation"] = (
    "Product review priority only; not evidence of alternative-factory benefit or feasibility"
)

product_scores.to_csv(REPORT_DIR / "task_10_10_balanced_scores.csv", index=False)
display(product_scores.sort_values("Balanced_Score", ascending=False))

,Product Name,Derived_Lead_Time_Mean_Days,Speed_Priority_Score,Score_Direction_x,Historical_Gross_Margin,Profit_Priority_Score,Score_Direction_y,Balanced_Score,Speed_Weight,Profit_Weight,Score_Interpretation
8,SweeTARTS,1383.500000,0.607029,Higher means greater review priority due to lo...,0.466667,0.460993,Higher means greater review priority due to lo...,0.534011,0.5,0.5,Product review priority only; not evidence of ...
3,Hair Toffee,1455.250000,1.000000,Higher means greater review priority due to lo...,0.777778,0.030733,Higher means greater review priority due to lo...,0.515366,0.5,0.5,Product review priority only; not evidence of ...
4,Kazookles,1272.989583,0.001769,Higher means greater review priority due to lo...,0.076923,1.000000,Higher means greater review priority due to lo...,0.500884,0.5,0.5,Product review priority only; not evidence of ...
5,Laffy Taffy,1383.100000,0.604838,Higher means greater review priority due to lo...,0.623116,0.244627,Higher means greater review priority due to lo...,0.424733,0.5,0.5,Product review priority only; not evidence of ...
6,Lickable Wallpaper,1339.138298,0.364062,Higher means greater review priority due to lo...,0.500000,0.414894,Higher means greater review priority due to lo...,0.389478,0.5,0.5,Product review priority only; not evidence of ...
0,Everlasting Gobstopper,1394.666667,0.668188,Higher means greater review priority due to lo...,0.800000,0.000000,Higher means greater review priority due to lo...,0.334094,0.5,0.5,Product review priority only; not evidence of ...
14,Wonka Gum,1306.525000,0.185440,Higher means greater review priority due to lo...,0.520000,0.387234,Higher means greater review priority due to lo...,0.286337,0.5,0.5,Product review priority only; not evidence of ...
2,Fun Dip,1272.666667,0.000000,Higher means greater review priority due to lo...,0.400000,0.553191,Higher means greater review priority due to lo...,0.276596,0.5,0.5,Product review priority only; not evidence of ...
12,Wonka Bar - Triple Dazzle Caramel,1325.661042,0.290248,Higher means greater review priority due to lo...,0.653333,0.202837,Higher means greater review priority due to lo...,0.246542,0.5,0.5,Product review priority only; not evidence of ...
7,Nerds,1274.000000,0.007303,Higher means greater review priority due to lo...,0.466667,0.460993,Higher means greater review priority due to lo...,0.234148,0.5,0.5,Product review priority only; not evidence of ...


In [48]:
# Create speed-versus-profit weight controls

# Define editable speed-first, balanced, and profit-protection weight profiles

weight_profiles = pd.DataFrame([
    ["Speed-first", 0.80, 0.20],
    ["Balanced", 0.50, 0.50],
    ["Profit-protection", 0.20, 0.80]
], columns=["Weight_Profile", "Speed_Weight", "Profit_Weight"])

weight_profiles["Weights_Sum"] = (
    weight_profiles["Speed_Weight"] + weight_profiles["Profit_Weight"]
)
weight_profiles["Valid_Weights"] = (
    (weight_profiles["Speed_Weight"] >= 0)
    & (weight_profiles["Profit_Weight"] >= 0)
    & np.isclose(weight_profiles["Weights_Sum"], 1.0)
)

if not weight_profiles["Valid_Weights"].all():
    raise ValueError("One or more weight profiles are invalid.")

weight_profiles.to_csv(REPORT_DIR / "task_10_11_weight_profiles.csv", index=False)
display(weight_profiles)

,Weight_Profile,Speed_Weight,Profit_Weight,Weights_Sum,Valid_Weights
0,Speed-first,0.8,0.2,1.0,True
1,Balanced,0.5,0.5,1.0,True
2,Profit-protection,0.2,0.8,1.0,True


In [50]:
# Rank alternative options for screening

# Rank product-factory rows by product review priority without inventing factory differences

ranked_options = factory_options.merge(
    product_scores[[
        "Product Name", "Speed_Priority_Score",
        "Profit_Priority_Score", "Balanced_Score"
    ]],
    on="Product Name",
    how="left"
)

ranked_options["Screening_Rank"] = (
    ranked_options["Balanced_Score"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
ranked_options["Factory_Option_Rank_Status"] = np.where(
    ranked_options["Is_Reference_Mapping"],
    "Product-level screening rank only",
    "Not a feasibility or factory-benefit rank"
)
ranked_options["Recommendation_Status"] = np.where(
    ranked_options["Is_Reference_Mapping"],
    "Reference mapping for review",
    "UNVERIFIED - do not treat as an approved alternative"
)

ranked_options = ranked_options.sort_values(
    ["Balanced_Score", "Product Name", "Factory Option"],
    ascending=[False, True, True]
)
ranked_options.to_csv(REPORT_DIR / "task_10_12_screening_ranking.csv", index=False)
display(ranked_options.head(25))

,Product Name,Reference Factory,Factory Option,Is_Reference_Mapping,Option_Type,Compatibility_Status,Factory_Specific_Lead_Time_Estimable,Factory_Specific_Cost_Available,Capacity_Data_Available,Speed_Priority_Score,Profit_Priority_Score,Balanced_Score,Screening_Rank,Factory_Option_Rank_Status,Recommendation_Status
40,SweeTARTS,Sugar Shack,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,0.607029,0.460993,0.534011,1,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
41,SweeTARTS,Sugar Shack,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,0.607029,0.460993,0.534011,1,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
42,SweeTARTS,Sugar Shack,Sugar Shack,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False,0.607029,0.460993,0.534011,1,Product-level screening rank only,Reference mapping for review
43,SweeTARTS,Sugar Shack,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,0.607029,0.460993,0.534011,1,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
44,SweeTARTS,Sugar Shack,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False,0.607029,0.460993,0.534011,1,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
15,Hair Toffee,The Other Factory,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1.000000,0.030733,0.515366,6,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
16,Hair Toffee,The Other Factory,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1.000000,0.030733,0.515366,6,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
17,Hair Toffee,The Other Factory,Sugar Shack,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1.000000,0.030733,0.515366,6,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
18,Hair Toffee,The Other Factory,The Other Factory,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False,1.000000,0.030733,0.515366,6,Product-level screening rank only,Reference mapping for review
19,Hair Toffee,The Other Factory,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False,1.000000,0.030733,0.515366,6,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...


# Top-N priorities, explanations, coverage, and warnings

## Generate Top 3 product review priorities

In [55]:
# Select the three highest-priority products for business review

top_3_products = product_scores.sort_values(
    ["Balanced_Score", "Product Name"],
    ascending=[False, True]
).head(3).copy()

top_3_products.insert(0, "Review_Priority_Rank", range(1, len(top_3_products) + 1))
top_3_products["Result_Type"] = "Top 3 product review priorities - not factory recommendations"
top_3_products.to_csv(REPORT_DIR / "task_10_13_top_3_products.csv", index=False)
display(top_3_products)

,Review_Priority_Rank,Product Name,Derived_Lead_Time_Mean_Days,Speed_Priority_Score,Score_Direction_x,Historical_Gross_Margin,Profit_Priority_Score,Score_Direction_y,Balanced_Score,Speed_Weight,Profit_Weight,Score_Interpretation,Result_Type
8,1,SweeTARTS,1383.500000,0.607029,Higher means greater review priority due to lo...,0.466667,0.460993,Higher means greater review priority due to lo...,0.534011,0.5,0.5,Product review priority only; not evidence of ...,Top 3 product review priorities - not factory ...
3,2,Hair Toffee,1455.250000,1.000000,Higher means greater review priority due to lo...,0.777778,0.030733,Higher means greater review priority due to lo...,0.515366,0.5,0.5,Product review priority only; not evidence of ...,Top 3 product review priorities - not factory ...
4,3,Kazookles,1272.989583,0.001769,Higher means greater review priority due to lo...,0.076923,1.000000,Higher means greater review priority due to lo...,0.500884,0.5,0.5,Product review priority only; not evidence of ...,Top 3 product review priorities - not factory ...


In [57]:
# Generate Top 5 product review priorities

# Select the five highest-priority products for business review

top_5_products = product_scores.sort_values(
    ["Balanced_Score", "Product Name"],
    ascending=[False, True]
).head(5).copy()

top_5_products.insert(0, "Review_Priority_Rank", range(1, len(top_5_products) + 1))
top_5_products["Result_Type"] = "Top 5 product review priorities - not factory recommendations"
top_5_products.to_csv(REPORT_DIR / "task_10_14_top_5_products.csv", index=False)
display(top_5_products)

,Review_Priority_Rank,Product Name,Derived_Lead_Time_Mean_Days,Speed_Priority_Score,Score_Direction_x,Historical_Gross_Margin,Profit_Priority_Score,Score_Direction_y,Balanced_Score,Speed_Weight,Profit_Weight,Score_Interpretation,Result_Type
8,1,SweeTARTS,1383.500000,0.607029,Higher means greater review priority due to lo...,0.466667,0.460993,Higher means greater review priority due to lo...,0.534011,0.5,0.5,Product review priority only; not evidence of ...,Top 5 product review priorities - not factory ...
3,2,Hair Toffee,1455.250000,1.000000,Higher means greater review priority due to lo...,0.777778,0.030733,Higher means greater review priority due to lo...,0.515366,0.5,0.5,Product review priority only; not evidence of ...,Top 5 product review priorities - not factory ...
4,3,Kazookles,1272.989583,0.001769,Higher means greater review priority due to lo...,0.076923,1.000000,Higher means greater review priority due to lo...,0.500884,0.5,0.5,Product review priority only; not evidence of ...,Top 5 product review priorities - not factory ...
5,4,Laffy Taffy,1383.100000,0.604838,Higher means greater review priority due to lo...,0.623116,0.244627,Higher means greater review priority due to lo...,0.424733,0.5,0.5,Product review priority only; not evidence of ...,Top 5 product review priorities - not factory ...
6,5,Lickable Wallpaper,1339.138298,0.364062,Higher means greater review priority due to lo...,0.500000,0.414894,Higher means greater review priority due to lo...,0.389478,0.5,0.5,Product review priority only; not evidence of ...,Top 5 product review priorities - not factory ...


In [59]:
# Generate Top 10 product review priorities

# Select the ten highest-priority products for business review

top_10_products = product_scores.sort_values(
    ["Balanced_Score", "Product Name"],
    ascending=[False, True]
).head(10).copy()

top_10_products.insert(0, "Review_Priority_Rank", range(1, len(top_10_products) + 1))
top_10_products["Result_Type"] = "Top 10 product review priorities - not factory recommendations"
top_10_products.to_csv(REPORT_DIR / "task_10_15_top_10_products.csv", index=False)
display(top_10_products)

,Review_Priority_Rank,Product Name,Derived_Lead_Time_Mean_Days,Speed_Priority_Score,Score_Direction_x,Historical_Gross_Margin,Profit_Priority_Score,Score_Direction_y,Balanced_Score,Speed_Weight,Profit_Weight,Score_Interpretation,Result_Type
8,1,SweeTARTS,1383.500000,0.607029,Higher means greater review priority due to lo...,0.466667,0.460993,Higher means greater review priority due to lo...,0.534011,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
3,2,Hair Toffee,1455.250000,1.000000,Higher means greater review priority due to lo...,0.777778,0.030733,Higher means greater review priority due to lo...,0.515366,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
4,3,Kazookles,1272.989583,0.001769,Higher means greater review priority due to lo...,0.076923,1.000000,Higher means greater review priority due to lo...,0.500884,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
5,4,Laffy Taffy,1383.100000,0.604838,Higher means greater review priority due to lo...,0.623116,0.244627,Higher means greater review priority due to lo...,0.424733,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
6,5,Lickable Wallpaper,1339.138298,0.364062,Higher means greater review priority due to lo...,0.500000,0.414894,Higher means greater review priority due to lo...,0.389478,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
0,6,Everlasting Gobstopper,1394.666667,0.668188,Higher means greater review priority due to lo...,0.800000,0.000000,Higher means greater review priority due to lo...,0.334094,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
14,7,Wonka Gum,1306.525000,0.185440,Higher means greater review priority due to lo...,0.520000,0.387234,Higher means greater review priority due to lo...,0.286337,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
2,8,Fun Dip,1272.666667,0.000000,Higher means greater review priority due to lo...,0.400000,0.553191,Higher means greater review priority due to lo...,0.276596,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
12,9,Wonka Bar - Triple Dazzle Caramel,1325.661042,0.290248,Higher means greater review priority due to lo...,0.653333,0.202837,Higher means greater review priority due to lo...,0.246542,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...
7,10,Nerds,1274.000000,0.007303,Higher means greater review priority due to lo...,0.466667,0.460993,Higher means greater review priority due to lo...,0.234148,0.5,0.5,Product review priority only; not evidence of ...,Top 10 product review priorities - not factory...


In [65]:
# Generate recommendation explanations while preventing duplicate-column suffix conflicts

# Select only the required columns from each source.
score_columns = [
    "Product Name",
    "Speed_Priority_Score",
    "Profit_Priority_Score",
    "Balanced_Score"
]

baseline_columns = [
    "Product Name",
    "Order_Line_Count",
    "Historical_Sales",
    "Historical_Gross_Profit",
    "Historical_Gross_Margin",
    "Derived_Lead_Time_Mean_Days"
]

variability_columns = [
    "Product Name",
    "Lead_Time_Std_Days",
    "Lead_Time_IQR_Days"
]

# Rebuild the explanation dataframe using clean, non-overlapping columns.
explanations = (
    product_scores[score_columns]
    .merge(
        baseline[baseline_columns],
        on="Product Name",
        how="left"
    )
    .merge(
        variability[variability_columns],
        on="Product Name",
        how="left"
    )
)

# Calculate reference medians.
median_lead_time = explanations["Derived_Lead_Time_Mean_Days"].median()
median_margin = explanations["Historical_Gross_Margin"].median()
median_variability = explanations["Lead_Time_Std_Days"].median()

def explain_product(row):
    reasons = []

    lead_time = row["Derived_Lead_Time_Mean_Days"]
    margin = row["Historical_Gross_Margin"]
    variability_value = row["Lead_Time_Std_Days"]

    if pd.notna(lead_time) and pd.notna(median_lead_time):
        if lead_time > median_lead_time:
            reasons.append("Higher-than-median derived lead time")
        else:
            reasons.append("At-or-below-median derived lead time")

    if pd.notna(margin) and pd.notna(median_margin):
        if margin < median_margin:
            reasons.append("Lower-than-median historical gross margin")
        else:
            reasons.append("At-or-above-median historical gross margin")

    if pd.notna(variability_value) and pd.notna(median_variability):
        if variability_value > median_variability:
            reasons.append("Higher-than-median historical variability")
        else:
            reasons.append("At-or-below-median historical variability")

    return (
        "; ".join(reasons)
        if reasons
        else "Insufficient numeric historical indicators"
    )

# Generate explanations.
explanations["Review_Priority_Reason"] = explanations.apply(
    explain_product, axis=1
)

explanations["Important_Qualification"] = (
    "Product-level review priority only. Derived date differences are not verified "
    "transit durations. Alternative factory compatibility, availability, lead-time "
    "improvement, and profitability benefits are not established."
)

explanations["Explanation_Data_Status"] = np.where(
    explanations["Review_Priority_Reason"].eq(
        "Insufficient numeric historical indicators"
    ),
    "Limited historical indicators",
    "Historical indicators available"
)

# Sort and export.
explanations = explanations.sort_values(
    "Balanced_Score",
    ascending=False
).reset_index(drop=True)

explanations.to_csv(
    REPORT_DIR / "task_10_16_recommendation_explanations.csv",
    index=False
)

display(
    explanations[[
        "Product Name",
        "Balanced_Score",
        "Review_Priority_Reason",
        "Important_Qualification",
        "Explanation_Data_Status"
    ]]
)

print("Products explained:", len(explanations))
print(
    "Products with historical indicators:",
    explanations["Explanation_Data_Status"].eq(
        "Historical indicators available"
    ).sum()
)
print("Task 10.16 explanation export completed.")

,Product Name,Balanced_Score,Review_Priority_Reason,Important_Qualification,Explanation_Data_Status
0,SweeTARTS,0.534011,Higher-than-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available
1,Hair Toffee,0.515366,Higher-than-median derived lead time; At-or-ab...,Product-level review priority only. Derived da...,Historical indicators available
2,Kazookles,0.500884,At-or-below-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available
3,Laffy Taffy,0.424733,Higher-than-median derived lead time; At-or-ab...,Product-level review priority only. Derived da...,Historical indicators available
4,Lickable Wallpaper,0.389478,Higher-than-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available
5,Everlasting Gobstopper,0.334094,Higher-than-median derived lead time; At-or-ab...,Product-level review priority only. Derived da...,Historical indicators available
6,Wonka Gum,0.286337,At-or-below-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available
7,Fun Dip,0.276596,At-or-below-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available
8,Wonka Bar - Triple Dazzle Caramel,0.246542,Higher-than-median derived lead time; At-or-ab...,Product-level review priority only. Derived da...,Historical indicators available
9,Nerds,0.234148,At-or-below-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available


Products explained: 15
Products with historical indicators: 15
Task 10.16 explanation export completed.


In [67]:
# Calculate recommendation coverage

# Calculate product, row, and alternative-option coverage metrics

eligible_count = int(product_eligibility["Reference_Mapping_Available"].sum())
source_product_count = int(product_eligibility["Product Name"].nunique())
alternative_count = int((~factory_options["Is_Reference_Mapping"]).sum())
unsupported_alternative_count = int(
    ((~factory_options["Is_Reference_Mapping"]) &
     (factory_options["Compatibility_Status"] == "UNVERIFIED")).sum()
)

coverage = pd.DataFrame([
    ["Source products", source_product_count, "Products present in the original CSV"],
    ["Products with reference mapping", eligible_count, "Products with a Phase 07 reference mapping"],
    ["Unmapped products", len(unmapped_products), "Products requiring mapping review"],
    ["Original source rows represented", original_row_count, "Original CSV rows; no row removal"],
    ["Generated alternative options", alternative_count, "Other reference factories paired for screening"],
    ["Supported alternative options", 0, "No alternative compatibility evidence was supplied"],
    ["Unsupported alternative options", unsupported_alternative_count, "Alternatives lacking compatibility evidence"],
], columns=["Coverage_Metric", "Count", "Interpretation"])

coverage["Coverage_Percent"] = np.where(
    source_product_count > 0,
    coverage["Count"] / source_product_count * 100,
    np.nan
)
coverage.loc[
    coverage["Coverage_Metric"] == "Original source rows represented",
    "Coverage_Percent"
] = 100.0

coverage.to_csv(REPORT_DIR / "task_10_17_coverage_kpis.csv", index=False)
display(coverage)

,Coverage_Metric,Count,Interpretation,Coverage_Percent
0,Source products,15,Products present in the original CSV,100.0
1,Products with reference mapping,15,Products with a Phase 07 reference mapping,100.0
2,Unmapped products,0,Products requiring mapping review,0.0
3,Original source rows represented,10194,Original CSV rows; no row removal,100.0
4,Generated alternative options,60,Other reference factories paired for screening,400.0
5,Supported alternative options,0,No alternative compatibility evidence was supp...,0.0
6,Unsupported alternative options,60,Alternatives lacking compatibility evidence,400.0


In [69]:
# Flag unsupported scenarios

# Create an explicit warning register for every unsupported alternative factory option

unsupported_scenarios = factory_options.loc[
    ~factory_options["Is_Reference_Mapping"]
].copy()

unsupported_scenarios["Warning_Code"] = "UNVERIFIED_ALTERNATIVE_FACTORY"
unsupported_scenarios["Warning_Reason"] = (
    "No verified product-factory compatibility, historical dispatch, factory capacity, "
    "inventory, production-time, freight, or factory-specific cost evidence is available."
)
unsupported_scenarios["Permitted_Use"] = "Exploratory screening and data collection only"
unsupported_scenarios["Prohibited_Claim"] = (
    "Do not claim feasibility, shorter lead time, lower cost, or higher profit."
)

unsupported_scenarios.to_csv(
    REPORT_DIR / "task_10_18_unsupported_scenarios.csv", index=False
)
display(unsupported_scenarios.head(20))
print("Unsupported alternative options:", len(unsupported_scenarios))

,Product Name,Reference Factory,Factory Option,Is_Reference_Mapping,Option_Type,Compatibility_Status,Factory_Specific_Lead_Time_Estimable,Factory_Specific_Cost_Available,Capacity_Data_Available,Warning_Code,Warning_Reason,Permitted_Use,Prohibited_Claim
0,Everlasting Gobstopper,Secret Factory,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
2,Everlasting Gobstopper,Secret Factory,Sugar Shack,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
3,Everlasting Gobstopper,Secret Factory,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
4,Everlasting Gobstopper,Secret Factory,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
5,Fizzy Lifting Drinks,Sugar Shack,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
6,Fizzy Lifting Drinks,Sugar Shack,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
8,Fizzy Lifting Drinks,Sugar Shack,The Other Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
9,Fizzy Lifting Drinks,Sugar Shack,Wicked Choccy's,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
10,Fun Dip,Sugar Shack,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
11,Fun Dip,Sugar Shack,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."


Unsupported alternative options: 60


# No-benefit test, weight sensitivity, and ranking QA



In [72]:
# Test the no-benefit scenario
# Verify that alternative factory options do not receive artificial lead-time or profit benefits

no_benefit_test = factory_options.copy()
no_benefit_test = no_benefit_test.merge(
    baseline[[
        "Product Name", "Derived_Lead_Time_Mean_Days",
        "Historical_Gross_Margin"
    ]],
    on="Product Name",
    how="left"
)

# With no validated factory effect or entered cost assumption, no alternative benefit is assigned.
no_benefit_test["Scenario_Lead_Time_Days"] = no_benefit_test["Derived_Lead_Time_Mean_Days"]
no_benefit_test["Scenario_Gross_Margin"] = no_benefit_test["Historical_Gross_Margin"]
no_benefit_test["Lead_Time_Improvement_Days"] = 0.0
no_benefit_test["Profit_Change_Estimate"] = 0.0
no_benefit_test["No_Benefit_Assumption"] = True

no_benefit_test["Test_Result"] = np.where(
    no_benefit_test["Lead_Time_Improvement_Days"].eq(0)
    & no_benefit_test["Profit_Change_Estimate"].eq(0),
    "PASS",
    "FAIL"
)

no_benefit_test.to_csv(REPORT_DIR / "task_10_19_no_benefit_test.csv", index=False)
display(no_benefit_test[[
    "Product Name", "Factory Option", "Is_Reference_Mapping",
    "Lead_Time_Improvement_Days", "Profit_Change_Estimate", "Test_Result"
]].head(20))


,Product Name,Factory Option,Is_Reference_Mapping,Lead_Time_Improvement_Days,Profit_Change_Estimate,Test_Result
0,Everlasting Gobstopper,Lot's O' Nuts,False,0.0,0.0,PASS
1,Everlasting Gobstopper,Secret Factory,True,0.0,0.0,PASS
2,Everlasting Gobstopper,Sugar Shack,False,0.0,0.0,PASS
3,Everlasting Gobstopper,The Other Factory,False,0.0,0.0,PASS
4,Everlasting Gobstopper,Wicked Choccy's,False,0.0,0.0,PASS
5,Fizzy Lifting Drinks,Lot's O' Nuts,False,0.0,0.0,PASS
6,Fizzy Lifting Drinks,Secret Factory,False,0.0,0.0,PASS
7,Fizzy Lifting Drinks,Sugar Shack,True,0.0,0.0,PASS
8,Fizzy Lifting Drinks,The Other Factory,False,0.0,0.0,PASS
9,Fizzy Lifting Drinks,Wicked Choccy's,False,0.0,0.0,PASS


In [74]:
# Run weight sensitivity analysis

# Recalculate product review rankings under speed-first, balanced, and profit-protection weights

sensitivity_frames = []

for _, profile in weight_profiles.iterrows():
    temp = product_scores[[
        "Product Name", "Speed_Priority_Score", "Profit_Priority_Score"
    ]].copy()

    temp["Weight_Profile"] = profile["Weight_Profile"]
    temp["Speed_Weight"] = profile["Speed_Weight"]
    temp["Profit_Weight"] = profile["Profit_Weight"]
    temp["Sensitivity_Score"] = (
        profile["Speed_Weight"] * temp["Speed_Priority_Score"]
        + profile["Profit_Weight"] * temp["Profit_Priority_Score"]
    )
    temp["Rank"] = temp["Sensitivity_Score"].rank(
        method="min", ascending=False
    ).astype("Int64")
    sensitivity_frames.append(temp)

weight_sensitivity = pd.concat(sensitivity_frames, ignore_index=True)

rank_pivot = weight_sensitivity.pivot(
    index="Product Name",
    columns="Weight_Profile",
    values="Rank"
).reset_index()

rank_columns = [c for c in rank_pivot.columns if c != "Product Name"]
rank_pivot["Rank_Range_Across_Profiles"] = (
    rank_pivot[rank_columns].max(axis=1) - rank_pivot[rank_columns].min(axis=1)
)

weight_sensitivity.to_csv(
    REPORT_DIR / "task_10_20_weight_sensitivity_detail.csv", index=False
)
rank_pivot.to_csv(
    REPORT_DIR / "task_10_20_weight_sensitivity_rank_movement.csv", index=False
)

display(rank_pivot.sort_values("Rank_Range_Across_Profiles", ascending=False))

Weight_Profile,Product Name,Balanced,Profit-protection,Speed-first,Rank_Range_Across_Profiles
0,Everlasting Gobstopper,6,15,3,12
4,Kazookles,3,1,12,11
2,Fun Dip,8,3,13,10
7,Nerds,10,5,14,9
3,Hair Toffee,2,8,1,7
11,Wonka Bar - Nutty Crunch Surprise,12,14,7,7
1,Fizzy Lifting Drinks,15,9,15,6
13,Wonka Bar -Scrumdiddlyumptious,14,13,8,6
12,Wonka Bar - Triple Dazzle Caramel,9,10,6,4
14,Wonka Gum,7,6,10,4


In [76]:
# Validate ranking consistency and QA rules

# Run consistency checks for rankings, unsupported options, no-benefit assumptions, and score ranges

qa_checks = []

def add_qa(name, condition, detail):
    qa_checks.append({
        "Check": name,
        "Passed": bool(condition),
        "Detail": detail
    })

add_qa(
    "All source products accounted for",
    set(product_eligibility["Product Name"]) == source_products,
    "Eligibility table includes every distinct source product."
)
add_qa(
    "No duplicate product-reference mappings",
    not mapping["Product Name"].duplicated().any(),
    "Each product has at most one reference mapping."
)
add_qa(
    "Weights sum to one",
    np.allclose(weight_profiles["Weights_Sum"], 1.0),
    "All defined weight profiles sum to 1."
)
add_qa(
    "Scores are within zero and one",
    product_scores[["Speed_Priority_Score", "Profit_Priority_Score", "Balanced_Score"]]
    .dropna().apply(lambda col: col.between(0, 1).all()).all(),
    "Normalized screening scores should remain between 0 and 1."
)
add_qa(
    "Alternative options remain unverified",
    (factory_options.loc[
        ~factory_options["Is_Reference_Mapping"], "Compatibility_Status"
    ] == "UNVERIFIED").all(),
    "No alternative is labelled as feasible or verified."
)
add_qa(
    "Alternative factory lead-time estimates remain unavailable",
    time_comparison.loc[
        ~time_comparison["Is_Reference_Mapping"],
        "Factory_Option_Lead_Time_Estimate_Days"
    ].isna().all(),
    "No unsupported factory-specific lead-time estimates are populated."
)
add_qa(
    "No-benefit test passes",
    (no_benefit_test["Test_Result"] == "PASS").all(),
    "No artificial lead-time or profit benefit is assigned."
)
add_qa(
    "Top 3 count is correct",
    len(top_3_products) == min(3, len(product_scores)),
    "Top 3 output has the expected number of products."
)
add_qa(
    "Top 5 count is correct",
    len(top_5_products) == min(5, len(product_scores)),
    "Top 5 output has the expected number of products."
)
add_qa(
    "Top 10 count is correct",
    len(top_10_products) == min(10, len(product_scores)),
    "Top 10 output has the expected number of products."
)
add_qa(
    "All alternative options included in warnings",
    len(unsupported_scenarios) == int((~factory_options["Is_Reference_Mapping"]).sum()),
    "Every alternative option is present in the unsupported-scenario register."
)
add_qa(
    "Ranking is deterministic",
    ranked_options[["Product Name", "Factory Option"]].equals(
        ranked_options.sort_values(
            ["Balanced_Score", "Product Name", "Factory Option"],
            ascending=[False, True, True]
        )[["Product Name", "Factory Option"]]
    ),
    "Ranking uses stable, explicit sort keys."
)

qa_report = pd.DataFrame(qa_checks)
qa_report.to_csv(REPORT_DIR / "task_10_21_ranking_consistency_qa.csv", index=False)

display(qa_report)
print("QA checks passed:", int(qa_report["Passed"].sum()))
print("QA checks failed:", int((~qa_report["Passed"]).sum()))

if not qa_report["Passed"].all():
    raise AssertionError("One or more Phase 10 ranking QA checks failed.")

,Check,Passed,Detail
0,All source products accounted for,True,Eligibility table includes every distinct sour...
1,No duplicate product-reference mappings,True,Each product has at most one reference mapping.
2,Weights sum to one,True,All defined weight profiles sum to 1.
3,Scores are within zero and one,True,Normalized screening scores should remain betw...
4,Alternative options remain unverified,True,No alternative is labelled as feasible or veri...
5,Alternative factory lead-time estimates remain...,True,No unsupported factory-specific lead-time esti...
6,No-benefit test passes,True,No artificial lead-time or profit benefit is a...
7,Top 3 count is correct,True,Top 3 output has the expected number of products.
8,Top 5 count is correct,True,Top 5 output has the expected number of products.
9,Top 10 count is correct,True,Top 10 output has the expected number of produ...


QA checks passed: 12
QA checks failed: 0


# Business action plan and final exports

In [79]:
# Prepare the business action plan

# Prepare evidence-based business actions with owners, supporting evidence, and required next data

action_plan = pd.DataFrame([
    {
        "Priority": 1,
        "Business_Action": "Investigate the unusually long derived Order Date to Ship Date differences.",
        "Evidence_From_Analysis": (
            "Historical date differences are derived directly from the original date fields; "
            "they are not verified transit durations."
        ),
        "Suggested_Owner": "Order Management / Data Governance",
        "Next_Data_Required": (
            "Confirm date-field definitions, order lifecycle milestones, shipment records, "
            "and whether dates represent planned or actual events."
        ),
        "Expected_Decision": "Determine whether derived lead-time days represent an operational KPI."
    },
    {
        "Priority": 2,
        "Business_Action": "Review products with the highest speed-priority scores.",
        "Evidence_From_Analysis": "Product-level historical derived lead-time ranking.",
        "Suggested_Owner": "Supply Chain Planning",
        "Next_Data_Required": (
            "Validated production start/end dates, dispatch dates, delivery dates, "
            "and product-level service targets."
        ),
        "Expected_Decision": "Identify products that merit operational investigation."
    },
    {
        "Priority": 3,
        "Business_Action": "Review products with lower historical gross margins.",
        "Evidence_From_Analysis": "Historical product-level gross margin and profit indicators.",
        "Suggested_Owner": "Finance / Commercial Analytics",
        "Next_Data_Required": (
            "Verified manufacturing cost, freight cost, discounts, rebates, returns, "
            "and cost allocation rules."
        ),
        "Expected_Decision": "Validate margin drivers before changing prices or sourcing."
    },
    {
        "Priority": 4,
        "Business_Action": "Validate product-to-factory compatibility before evaluating reassignment.",
        "Evidence_From_Analysis": (
            "The Phase 07 mapping is a reference mapping; alternative factory options are unverified."
        ),
        "Suggested_Owner": "Manufacturing / Production Engineering",
        "Next_Data_Required": (
            "Approved product-factory compatibility matrix, equipment capability, "
            "quality approvals, and production restrictions."
        ),
        "Expected_Decision": "Establish which alternative factory options are genuinely feasible."
    },
    {
        "Priority": 5,
        "Business_Action": "Collect factory capacity and production-time information.",
        "Evidence_From_Analysis": "No validated capacity or production-time information is available in the current dataset.",
        "Suggested_Owner": "Factory Operations",
        "Next_Data_Required": (
            "Available capacity, utilization, shifts, changeover times, production calendars, "
            "inventory, backlog, and throughput."
        ),
        "Expected_Decision": "Support future capacity-constrained allocation analysis."
    },
    {
        "Priority": 6,
        "Business_Action": "Collect actual factory-specific freight and manufacturing costs.",
        "Evidence_From_Analysis": "Actual alternative factory costs were not supplied; cost inputs remain hypothetical.",
        "Suggested_Owner": "Finance / Logistics",
        "Next_Data_Required": (
            "Actual lane-level freight invoices, shipment quantities, manufacturing costs, "
            "handling costs, and effective dates."
        ),
        "Expected_Decision": "Enable evidence-based total-cost comparisons."
    },
    {
        "Priority": 7,
        "Business_Action": "Re-run optimization only after operational evidence is available.",
        "Evidence_From_Analysis": (
            "Current scores rank product review priority and do not estimate factory-specific benefits."
        ),
        "Suggested_Owner": "Supply Chain Analytics",
        "Next_Data_Required": (
            "Verified factory assignment history, compatibility, capacity, lead-time, "
            "and cost records."
        ),
        "Expected_Decision": "Build a defensible factory allocation model with validation."
    }
])

action_plan.to_csv(REPORT_DIR / "task_10_22_business_action_plan.csv", index=False)
display(action_plan)

,Priority,Business_Action,Evidence_From_Analysis,Suggested_Owner,Next_Data_Required,Expected_Decision
0,1,Investigate the unusually long derived Order D...,Historical date differences are derived direct...,Order Management / Data Governance,"Confirm date-field definitions, order lifecycl...",Determine whether derived lead-time days repre...
1,2,Review products with the highest speed-priorit...,Product-level historical derived lead-time ran...,Supply Chain Planning,"Validated production start/end dates, dispatch...",Identify products that merit operational inves...
2,3,Review products with lower historical gross ma...,Historical product-level gross margin and prof...,Finance / Commercial Analytics,"Verified manufacturing cost, freight cost, dis...",Validate margin drivers before changing prices...
3,4,Validate product-to-factory compatibility befo...,The Phase 07 mapping is a reference mapping; a...,Manufacturing / Production Engineering,"Approved product-factory compatibility matrix,...",Establish which alternative factory options ar...
4,5,Collect factory capacity and production-time i...,No validated capacity or production-time infor...,Factory Operations,"Available capacity, utilization, shifts, chang...",Support future capacity-constrained allocation...
5,6,Collect actual factory-specific freight and ma...,Actual alternative factory costs were not supp...,Finance / Logistics,"Actual lane-level freight invoices, shipment q...",Enable evidence-based total-cost comparisons.
6,7,Re-run optimization only after operational evi...,Current scores rank product review priority an...,Supply Chain Analytics,"Verified factory assignment history, compatibi...",Build a defensible factory allocation model wi...


In [81]:
# Export recommendations to CSV and Excel

# Export Phase 10 outputs, assumptions, limitations, and QA results to a consolidated Excel workbook

ASSUMPTIONS = pd.DataFrame([
    ["Source data", "Original Nassau Candy Distributor CSV", "Original source is read-only; no source values are modified."],
    ["Historical baseline", "Order Date to Ship Date difference", "Called derived lead-time days; not verified transit duration."],
    ["Factory mapping", "Phase 07 product-factory reference mapping", "Not proof of actual historical factory dispatch."],
    ["Alternative factories", "Other factories listed in the reference mapping", "All alternative options are unverified until compatibility evidence is supplied."],
    ["Factory lead-time estimates", "Not calculated", "No validated factory-specific effect or historical factory assignment is available."],
    ["Profitability", "Historical Sales, Gross Profit, Cost, and Units", "Historical product profitability is not alternative-factory profitability."],
    ["Additional costs", "Blank user-editable hypothetical cost template", "No actual alternative factory costs were provided."],
    ["Scoring", "Normalized product-level speed and profit priorities", "Scores prioritize review; they do not prove feasibility or optimization."],
    ["Weights", "Speed-first 80/20, balanced 50/50, profit-protection 20/80", "Weights are transparent and can be changed."],
    ["No-benefit test", "Zero improvement assigned without evidence", "Prevents unsupported scenario benefits from appearing."],
    ["Distance and routing", "Not used to estimate factory benefits", "Reference coordinates or straight-line distance do not establish real routes or freight costs."]
], columns=["Area", "Method_or_Input", "Assumption_or_Limitation"])

LIMITATIONS = pd.DataFrame([
    ["Historical date differences", "The derived differences are unusually long and require semantic validation."],
    ["Actual factory assignment", "The dataset does not provide validated historical factory dispatch."],
    ["Factory compatibility", "Alternative product-factory compatibility is not verified."],
    ["Factory capacity", "Capacity, utilization, inventory, backlog, and throughput are unavailable."],
    ["Factory lead time", "Alternative factory lead-time effects are not estimable."],
    ["Factory profitability", "Alternative factory manufacturing and freight costs are unavailable."],
    ["Hypothetical costs", "User-entered incremental costs are assumptions, not actual cost records."],
    ["Recommendation scores", "Scores are product-level review priorities, not factory-specific benefit estimates."],
    ["Causality", "The analysis does not establish that reassignment causes shorter lead times or higher profit."]
], columns=["Limitation", "Explanation"])

# Build an Excel workbook with unique, Excel-compatible sheet names.
excel_file = REPORT_DIR / "Phase_10_Optimization_Recommendations.xlsx"

sheets = {
    "Eligibility": product_eligibility,
    "Factory Options": factory_options,
    "Historical Baseline": baseline,
    "Lead Time Compare": time_comparison,
    "Variability": variability,
    "Profitability": profitability,
    "Scoring Formula": scoring_formula,
    "Speed Scores": speed_scores,
    "Profit Scores": profit_scores,
    "Balanced Scores": product_scores,
    "Weight Profiles": weight_profiles,
    "All Rankings": ranked_options,
    "Top 3": top_3_products,
    "Top 5": top_5_products,
    "Top 10": top_10_products,
    "Explanations": explanations,
    "Coverage": coverage,
    "Unsupported Options": unsupported_scenarios,
    "No Benefit Test": no_benefit_test,
    "Weight Sensitivity": weight_sensitivity,
    "Rank Movement": rank_pivot,
    "Ranking QA": qa_report,
    "Business Action Plan": action_plan,
    "Assumptions": ASSUMPTIONS,
    "Limitations": LIMITATIONS,
    "Hypothetical Cost Input": hypothetical_cost_inputs
}

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    for sheet_name, data in sheets.items():
        data.to_excel(writer, sheet_name=sheet_name[:31], index=False)

# Export the principal recommendation tables separately for convenient reuse.
product_scores.to_csv(
    REPORT_DIR / "Phase_10_Product_Review_Priorities.csv", index=False
)
ranked_options.to_csv(
    REPORT_DIR / "Phase_10_Factory_Option_Screening.csv", index=False
)
action_plan.to_csv(
    REPORT_DIR / "Phase_10_Business_Action_Plan.csv", index=False
)
ASSUMPTIONS.to_csv(
    REPORT_DIR / "Phase_10_Assumptions.csv", index=False
)
LIMITATIONS.to_csv(
    REPORT_DIR / "Phase_10_Limitations.csv", index=False
)

print("Excel workbook saved:", excel_file)
print("Recommendation outputs saved in:", REPORT_DIR)
print("Hypothetical cost template:", HYPOTHETICAL_COST_FILE)


Excel workbook saved: C:\Users\hp\Nassau_Candy_Final_Submission\reports\optimization_recommendations\Phase_10_Optimization_Recommendations.xlsx
Recommendation outputs saved in: C:\Users\hp\Nassau_Candy_Final_Submission\reports\optimization_recommendations
Hypothetical cost template: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\phase10_hypothetical_cost_inputs_template.csv


In [91]:
# Verify  outputs, original CSV integrity, row counts, and unsupported-option safeguards

ORIGINAL_HASH_AFTER = sha256_file(RAW_FILE)

expected_files = [
    REPORT_DIR / "Phase_10_Optimization_Recommendations.xlsx",
    REPORT_DIR / "Phase_10_Product_Review_Priorities.csv",
    REPORT_DIR / "Phase_10_Factory_Option_Screening.csv",
    REPORT_DIR / "Phase_10_Business_Action_Plan.csv",
    REPORT_DIR / "Phase_10_Assumptions.csv",
    REPORT_DIR / "Phase_10_Limitations.csv",
    HYPOTHETICAL_COST_FILE
]

integrity_checks = [
    ("Original CSV exists", RAW_FILE.exists()),
    ("Original row count preserved in analysis copy", len(df_phase10) == original_row_count),
    ("Original columns retained", all(c in df_phase10.columns for c in original_columns)),
    ("Original column order retained", df_phase10[original_columns].columns.tolist() == original_columns),
    ("Original CSV hash unchanged", ORIGINAL_HASH_BEFORE == ORIGINAL_HASH_AFTER),
    ("Original source rows represented in baseline", int(baseline["Order_Line_Count"].sum()) == original_row_count),
    ("All source products appear in eligibility table", set(product_eligibility["Product Name"]) == source_products),
    ("No alternative options labelled verified", (
        factory_options.loc[~factory_options["Is_Reference_Mapping"], "Compatibility_Status"]
        .eq("UNVERIFIED").all()
    )),
    ("Alternative factory lead-time estimates not invented", (
        time_comparison.loc[
            ~time_comparison["Is_Reference_Mapping"],
            "Factory_Option_Lead_Time_Estimate_Days"
        ].isna().all()
    )),
    ("No-benefit test passed", no_benefit_test["Test_Result"].eq("PASS").all()),
    ("Ranking QA passed", qa_report["Passed"].all()),
    ("Excel workbook exists", excel_file.exists()),
    ("Hypothetical cost template exists", HYPOTHETICAL_COST_FILE.exists()),
    ("All expected output files exist", all(path.exists() for path in expected_files))
]

integrity_report = pd.DataFrame(
    integrity_checks, columns=["Integrity_Check", "Passed"]
)
integrity_report["Status"] = np.where(
    integrity_report["Passed"], "PASS", "FAIL"
)

integrity_report.to_csv(
    REPORT_DIR / "Phase_10_Integrity_Report.csv", index=False
)

display(integrity_report)

print("\nPHASE 10 INTEGRITY SUMMARY")
print("Total checks:", len(integrity_report))
print("Passed:", int(integrity_report["Passed"].sum()))
print("Failed:", int((~integrity_report["Passed"]).sum()))
print("Original CSV SHA-256 before:", ORIGINAL_HASH_BEFORE)
print("Original CSV SHA-256 after: ", ORIGINAL_HASH_AFTER)

if integrity_report["Passed"].all():
    print("\nSUCCESS: integrity verification completed.")
else:
    print("\nWARNING: Some integrity checks failed. Review the failed rows before marking Phase 10 complete.")

,Integrity_Check,Passed,Status
0,Original CSV exists,True,PASS
1,Original row count preserved in analysis copy,True,PASS
2,Original columns retained,True,PASS
3,Original column order retained,True,PASS
4,Original CSV hash unchanged,True,PASS
5,Original source rows represented in baseline,True,PASS
6,All source products appear in eligibility table,True,PASS
7,No alternative options labelled verified,True,PASS
8,Alternative factory lead-time estimates not in...,True,PASS
9,No-benefit test passed,True,PASS



PHASE 10 INTEGRITY SUMMARY
Total checks: 14
Passed: 14
Failed: 0
Original CSV SHA-256 before: 6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f
Original CSV SHA-256 after:  6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f

SUCCESS: integrity verification completed.


In [93]:
# Inspect actual Phase 05, Phase 07, and Phase 10 files and their schemas before Phase 11 implementation

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\hp\Nassau_Candy_Final_Submission")

inspection_folders = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "reports" / "optimization_recommendations",
    PROJECT_ROOT / "reports" / "profitability_sensitivity"
]

for folder in inspection_folders:
    print("\n" + "=" * 90)
    print("FOLDER:", folder)
    print("EXISTS:", folder.exists())

    if not folder.exists():
        continue

    files = sorted(
        [p for p in folder.iterdir() if p.is_file()],
        key=lambda p: p.name.lower()
    )

    for file_path in files:
        print("FILE:", file_path.name)

        if file_path.suffix.lower() == ".csv":
            try:
                sample = pd.read_csv(file_path, nrows=5)
                print("  COLUMNS:", sample.columns.tolist())
                print("  SAMPLE ROWS:", len(sample))
            except Exception as error:
                print("  CSV READ ERROR:", str(error))

        elif file_path.suffix.lower() in [".pkl", ".joblib", ".json"]:
            print("  FILE TYPE:", file_path.suffix)

# Identify likely model and recommendation files recursively.
print("\n" + "=" * 90)
print("RECURSIVE MODEL AND RECOMMENDATION FILE DISCOVERY")

for folder in [
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "data" / "processed"
]:
    if folder.exists():
        for file_path in sorted(folder.rglob("*")):
            if file_path.is_file() and any(
                word in file_path.name.lower()
                for word in ["model", "metric", "evaluation", "recommend",
                             "scenario", "mapping", "factory", "lead_time"]
            ):
                print(file_path.relative_to(PROJECT_ROOT))


FOLDER: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed
EXISTS: True
FILE: advanced_factory_distance_comparison.csv
  COLUMNS: ['Product Name', 'Baseline Factory', 'Candidate Factory', 'Scenario Type', 'Straight_Line_Distance_KM', 'Distance Type']
  SAMPLE ROWS: 5
FILE: advanced_product_factory_scenario_matrix.csv
  COLUMNS: ['Product Name', 'Baseline Factory', 'Candidate Factory', 'Scenario Type', 'Factory Compatibility', 'Scenario Classification', 'Compatibility Evidence']
  SAMPLE ROWS: 5
FILE: Clustering_Product_Aggregation.csv
  COLUMNS: ['Product Name', 'Records', 'Unique_Orders', 'Sales_Total', 'Sales_Mean', 'Units_Total', 'Units_Mean', 'Gross_Profit_Total', 'Gross_Profit_Mean', 'Cost_Total', 'Cost_Mean', 'Lead_Time_Median', 'Lead_Time_Mean', 'Lead_Time_Std', 'Profit_Margin_Pct', 'Sales_Per_Unit', 'Gross_Profit_Per_Unit']
  SAMPLE ROWS: 5
FILE: Clustering_Product_Cluster_Profiles.csv
  COLUMNS: ['Cluster', 'Records', 'Sales_Total', 'Units_Total', 'Gross_Profit_Total', 

In [95]:
# Inspect model metadata, KPI source values, and Phase 10 support evidence

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\hp\Nassau_Candy_Final_Submission")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
PHASE10_DIR = PROJECT_ROOT / "reports" / "optimization_recommendations"

# Inspect saved model metadata
metadata_file = MODEL_DIR / "Nassau_Lead_Time_Model_Metadata.json"

print("=" * 90)
print("SAVED MODEL METADATA")

if metadata_file.exists():
    with open(metadata_file, "r", encoding="utf-8") as file:
        metadata = json.load(file)
    print(json.dumps(metadata, indent=2, default=str))
else:
    print("Metadata file not found.")

# Inspect selected CSV values and schemas
inspection_files = {
    "ML_Final_Model_Comparison": PROCESSED_DIR / "ML_Final_Model_Comparison.csv",
    "ML_Cross_Validation": PROCESSED_DIR / "ML_Cross_Validation.csv",
    "ML_Predictions": PROCESSED_DIR / "ML_Predictions.csv",
    "ML_LeadTime_Ranges": PROCESSED_DIR / "ML_LeadTime_Ranges.csv",
    "margin_stability_proxy": PROCESSED_DIR / "margin_stability_proxy.csv",
    "profit_variability_report": PROCESSED_DIR / "profit_variability_report.csv",
    "Phase_10_Product_Review_Priorities": PHASE10_DIR / "Phase_10_Product_Review_Priorities.csv",
    "Phase_10_Factory_Option_Screening": PHASE10_DIR / "Phase_10_Factory_Option_Screening.csv",
    "Phase_10_Coverage": PHASE10_DIR / "task_10_17_coverage_kpis.csv",
    "Phase_10_Unsupported_Scenarios": PHASE10_DIR / "task_10_18_unsupported_scenarios.csv",
    "Phase_10_Lead_Time_Comparison": PHASE10_DIR / "task_10_04_lead_time_comparison.csv"
}

for label, file_path in inspection_files.items():
    print("\n" + "=" * 90)
    print(label)
    print("PATH:", file_path)
    print("EXISTS:", file_path.exists())

    if not file_path.exists():
        continue

    try:
        data = pd.read_csv(file_path)
        print("SHAPE:", data.shape)
        print("COLUMNS:", data.columns.tolist())
        print("SAMPLE:")
        print(data.head(8).to_string(index=False))
    except Exception as error:
        print("READ ERROR:", repr(error))

# Inspect original source hash without changing the file
import hashlib

raw_file = PROJECT_ROOT / "data" / "raw" / "Nassau Candy Distributor.csv"

print("\n" + "=" * 90)
print("ORIGINAL SOURCE FILE CHECK")

if raw_file.exists():
    sha256 = hashlib.sha256()
    with open(raw_file, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    print("EXISTS:", True)
    print("SHA256:", sha256.hexdigest())
else:
    print("Original CSV not found.")

SAVED MODEL METADATA
{
  "project": "Nassau Candy Distributor",
  "target_variable": "Lead_Time_Days",
  "selected_model": "Tuned Random Forest",
  "predictor_features": [
    "Ship Mode",
    "Country/Region",
    "City",
    "State/Province",
    "Division",
    "Region",
    "Product ID",
    "Product Name",
    "Units",
    "Sales",
    "Gross Profit",
    "Cost",
    "Order_Year",
    "Order_Month",
    "Order_Quarter",
    "Order_Weekday",
    "Profit_Margin_Pct",
    "Sales_Per_Unit",
    "Cost_Per_Unit",
    "Profit_Per_Unit"
  ],
  "training_records": 8132,
  "testing_records": 2062,
  "test_MAE": 95.97949651478011,
  "test_RMSE": 120.7382358245334,
  "test_R2": 0.7929486715895369,
  "model_file": "Nassau_Lead_Time_Model.joblib"
}

ML_Final_Model_Comparison
PATH: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\ML_Final_Model_Comparison.csv
EXISTS: True
SHAPE: (4, 4)
COLUMNS: ['Model', 'Test_MAE', 'Test_RMSE', 'Test_R2']
SAMPLE:
              Model   Test_MAE  Test_RMS